## Import packages

In [2]:
# Needed this for loading the dataset locally
# from dotenv import load_dotenv
# load_dotenv()

import pandas as pd
from datasets import load_dataset
import numpy as np

from collections import Counter

import nltk

In [3]:
# dataset = load_dataset("coastalcph/tydi_xor_rc")

## Load Dataset

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")

## Filter and split Dataset

In [5]:
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()



#filter data 
langlst = ['ko','ar','te']
# Source - https://stackoverflow.com/a/59275490
# Posted by atinjanki
# Retrieved 2026-09-05, License - CC BY-SA 4.0

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]



In [6]:
# --------------------------- What is this for? --------------------------- 

# df_train_ar.duplicated().value_counts()
# df_train_ko.duplicated().value_counts()
# df_train_te.duplicated().value_counts()

print(f"{df_train_ar.duplicated().value_counts()}")
print(f"{df_train_ko.duplicated().value_counts()}")
print(f"{df_train_te.duplicated().value_counts()}")

# --------------------------- What is this for? --------------------------- 

False    2558
Name: count, dtype: int64
False    2412
True       10
Name: count, dtype: int64
False    1355
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
## CHANGE TO NLTK AT SOME POINT :)


def get_statistics(df, split, language):
    question_lengths = []
    context_lengths = []

    for question in df["question"]:
        question_lengths.append(len(tokenizer.tokenize(question)))
    
    for context in df["context"]:
        context_lengths.append(len(tokenizer.tokenize(context)))

    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])


    return {
            "split": split,
            "language": language,
            "n": len(df),
            "answerable_%": 100 * answerable_n / len(df),
            "unanswerable_%": 100 * unanswerable_n / len(df),
            "question_median": np.median(question_lengths),
            "question_IQR": np.percentile(question_lengths, 75) - np.percentile(question_lengths, 25),
            "context_median": np.median(context_lengths),
            "context_IQR": np.percentile(context_lengths, 75) - np.percentile(context_lengths, 25)
        }


statistics = pd.DataFrame([
    get_statistics(df_train_ar, "train", "ar"),
    get_statistics(df_train_ko, "train", "ko"),
    get_statistics(df_train_te, "train", "te"),
    get_statistics(df_validation_ar, "validation", "ar"),
    get_statistics(df_validation_ko, "validation", "ko"),
    get_statistics(df_validation_te, "validation", "te")
])

statistics.round(1)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


,split,language,n,answerable_%,unanswerable_%,question_median,question_IQR,context_median,context_IQR
0,train,ar,2558,90.0,10.0,13.0,6.0,123.0,94.0
1,train,ko,2422,97.4,2.6,14.0,4.0,115.0,87.0
2,train,te,1355,96.7,3.3,18.0,6.5,114.0,87.0
3,validation,ar,415,87.5,12.5,12.0,6.0,118.0,94.5
4,validation,ko,356,94.7,5.3,14.0,4.0,112.5,97.0
5,validation,te,384,75.8,24.2,19.5,6.0,142.0,88.2


In [8]:
overall_rows = []

for split, df in [("train", df_train_filtered), ("validation", df_validation_filtered)]:
    answerable_n = len(df[df["answerable"] == True])
    unanswerable_n = len(df[df["answerable"] == False])

    overall_rows.append({
        "split": split,
        "n": len(df),
        "answerable_%": 100 * answerable_n / len(df),
        "unanswerable_%": 100 * unanswerable_n / len(df)
    })

pd.DataFrame(overall_rows).round(1)

,split,n,answerable_%,unanswerable_%
0,train,6335,94.3,5.7
1,validation,1155,85.8,14.2


In [9]:
data_quality_rows = []

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te)
]

for split, language, df in dataframes:
    missing_values = df.isna().sum().sum()
    duplicate_pairs = df[["question", "context"]].duplicated().sum()

    data_quality_rows.append({
        "split": split,
        "language": language,
        "missing_values": missing_values,
        "duplicate_question_context_pairs": duplicate_pairs
    })

pd.DataFrame(data_quality_rows)


,split,language,missing_values,duplicate_question_context_pairs
0,train,ar,2558,0
1,train,ko,2422,10
2,train,te,1305,0
3,validation,ar,415,0
4,validation,ko,356,0
5,validation,te,284,0


In [10]:
def most_common_tokens(df):
    tokens = []

    for question in df["question"]:
        tokens.extend(tokenizer.tokenize(question))

    return Counter(tokens).most_common(5)


ar_top5 = most_common_tokens(df_train_ar)
ko_top5 = most_common_tokens(df_train_ko)
te_top5 = most_common_tokens(df_train_te)

print("Arabic:", ar_top5)
print("Korean:", ko_top5)
print("Telugu:", te_top5)


Arabic: [('؟', 2556), ('ال', 2269), ('م', 892), ('في', 624), ('من', 616)]
Korean: [('?', 2420), ('##가', 2219), ('##인', 1495), ('##는', 988), ('##은', 949)]
Telugu: [('?', 1355), ('ఎ', 882), ('##వ', 544), ('##ా', 515), ('ప', 505)]


In [11]:
span_rows = []

df_train_selected = df_train[df_train["lang"].isin(["ar", "ko", "te"])]
df_validation_selected = df_validation[df_validation["lang"].isin(["ar", "ko", "te"])]

dataframes = [
    ("train", "ar", df_train_ar),
    ("train", "ko", df_train_ko),
    ("train", "te", df_train_te),
    ("train", "all", df_train_selected),
    ("validation", "ar", df_validation_ar),
    ("validation", "ko", df_validation_ko),
    ("validation", "te", df_validation_te),
    ("validation", "all", df_validation_selected)
]

for split, language, df in dataframes:
    answerable = df[df["answerable"] == True]
    failures = 0

    for context, start, answer in zip(
        answerable["context"],
        answerable["answer_start"],
        answerable["answer"]
    ):
        start = int(start)

        if context[start:start + len(answer)] != answer:
            failures += 1

    span_rows.append({
        "split": split,
        "language": language,
        "checked": len(answerable),
        "failures": failures
    })

pd.DataFrame(span_rows)


,split,language,checked,failures
0,train,ar,2303,0
1,train,ko,2359,0
2,train,te,1310,0
3,train,all,5972,0
4,validation,ar,363,0
5,validation,ko,337,0
6,validation,te,291,0
7,validation,all,991,0


In [12]:
df_train_selected = df_train[df_train["lang"].isin(["ar", "ko", "te"])]

answerable_n = len(df_train_selected[df_train_selected["answerable"] == True])
unanswerable_n = len(df_train_selected[df_train_selected["answerable"] == False])

print("Answerable:", answerable_n)
print("Unanswerable:", unanswerable_n)

if answerable_n >= unanswerable_n:
    majority_label = True
else:
    majority_label = False

print("Majority prediction:", majority_label)



Answerable: 5972
Unanswerable: 363
Majority prediction: True


In [13]:
# Rule-based baseline: no question numbers, or at least one matching number.
import re

def extract_numbers(text):
    # int() converts each Unicode decimal digit to its ASCII equivalent.
    return {''.join(str(int(digit)) for digit in number)
            for number in re.findall(r'\d+', text)}

def rule_answerable(question, context):
    question_numbers = extract_numbers(question)
    context_numbers = extract_numbers(context)
    return not question_numbers or bool(question_numbers & context_numbers)

rule_predictions = [
    rule_answerable(question, context)
    for question, context in zip(
        df_validation_filtered['question'], df_validation_filtered['context']
    )
]

bin_list = [1 if elm else 0 for elm in rule_predictions]
print(sum(bin_list))
print(sum(bin_list)/len(bin_list))

1083
0.9376623376623376


In [14]:
from sklearn.metrics import classification_report

# Evaluate both baselines on the same validation examples.
evaluation = df_validation_filtered.copy()
evaluation['majority'] = majority_label
evaluation['number_overlap'] = rule_predictions

# 'macro avg' F1 is the mean of the two class F1 scores.
# Undefined precision/recall/F1 is reported as zero.
for language in ['all', 'ar', 'ko', 'te']:
    group = evaluation if language == 'all' else evaluation[evaluation['lang'] == language]
    for baseline in ['majority', 'number_overlap']:
        print(f'\n{baseline} | {language} | n={len(group)}')
        print(classification_report(
            group['answerable'], group[baseline],
            labels=[False, True], target_names=['unanswerable', 'answerable'],
            digits=3, zero_division=0
        ))



majority | all | n=1155
              precision    recall  f1-score   support

unanswerable      0.000     0.000     0.000       164
  answerable      0.858     1.000     0.924       991

    accuracy                          0.858      1155
   macro avg      0.429     0.500     0.462      1155
weighted avg      0.736     0.858     0.792      1155


number_overlap | all | n=1155
              precision    recall  f1-score   support

unanswerable      0.153     0.067     0.093       164
  answerable      0.859     0.938     0.897       991

    accuracy                          0.815      1155
   macro avg      0.506     0.503     0.495      1155
weighted avg      0.758     0.815     0.783      1155


majority | ar | n=415
              precision    recall  f1-score   support

unanswerable      0.000     0.000     0.000        52
  answerable      0.875     1.000     0.933       363

    accuracy                          0.875       415
   macro avg      0.437     0.500     0.467      

In [15]:
false_positives = evaluation[evaluation['number_overlap'] & ~evaluation['answerable']]
false_negatives = evaluation[~evaluation['number_overlap'] & evaluation['answerable']]

print('Majority false positives:', sum(evaluation['majority'] & ~evaluation['answerable']))
print('Majority false negatives:', sum(~evaluation['majority'] & evaluation['answerable']))
print('Number-overlap false positives:', len(false_positives))
print('Number-overlap false negatives:', len(false_negatives))

columns = ['lang', 'question', 'context', 'answer']
display(false_positives[columns].head(2))
display(false_negatives[columns].head(2))


Majority false positives: 164
Majority false negatives: 0
Number-overlap false positives: 153
Number-overlap false negatives: 61


,lang,question,context,answer
364,ko,시차는 중력과 관련이 있는가?,Gravitational time dilation was first describe...,no
368,ko,맹장 없이 살 수 있을까?,"In modern humans, the vermiform appendix is a ...",no


,lang,question,context,answer
16,te,2010 నాటికి భారతదేశంలో క్రైస్తవులు ఎక్కువ ఉండే...,Even though Christians are a significant minor...,"Meghalaya, Mizoram, and Nagaland"
29,te,2018 వరకు బైబిలు ను ఎన్ని భాషలలో అనువదించారు?,The Bible has been translated into many langua...,"3,312 languages"
